# Week 8 – Day 2 | ExerciseXP
## Machine Learning: Problem Definition, Feature Selection, Model Evaluation & Design

---
## 🌟 Exercise 1 – Defining the Problem and Data Collection for Loan Default Prediction

### Problem Statement

**Goal:** Build a supervised binary classification model that predicts whether a loan applicant will **default** (fail to repay) or **not default** on a loan.

Loan defaults cause significant financial losses for lending institutions. Early and accurate identification of high-risk applicants allows lenders to adjust loan terms, require additional guarantees, or decline applications — reducing portfolio risk while remaining fair to creditworthy borrowers.

- **Target variable:** `Loan_Status` — binary (Default = 1, No Default = 0)
- **Type of problem:** Supervised binary classification
- **Evaluation horizon:** Pre-approval (predict at time of application)

### Data Types Needed

| Category | Features | Type |
|---|---|---|
| Applicant demographics | Age, Gender, Marital Status, Dependents, Education | Categorical / Numerical |
| Income & employment | Monthly income, Employment status, Self-employed flag, Years employed | Numerical / Categorical |
| Loan details | Loan amount requested, Loan term, Loan purpose | Numerical / Categorical |
| Credit history | Credit score, Number of open accounts, Missed payments, Bankruptcy history | Numerical / Binary |
| Assets | Property area, Owned assets value | Categorical / Numerical |
| Repayment history | Past loan repayment records, Credit card utilization | Numerical |
| Macro context | Interest rate at origination, Economic indicators (optional) | Numerical |

### Data Sources

1. **Financial institution's internal records** – Application forms, CRM systems, repayment transaction history. Most reliable source since it reflects actual customer behavior within the institution.
2. **Credit bureaus** (e.g., Experian, Equifax, TransUnion) – Provide credit scores, total debt, payment history, public records. Accessed via API at the time of application.
3. **Government / public records** – Property ownership, bankruptcy filings, court judgments. Available through open data portals.
4. **Open datasets for research** – e.g., [Loan Prediction Dataset on GitHub](https://github.com/devtlv/Datasets-GEN-AI-Bootcamp) used in this exercise.
5. **Applicant self-reported data** – Income, employer, existing liabilities declared on the application form (subject to verification).

**Data privacy considerations:** All data collection must comply with GDPR / local data protection regulations. Sensitive attributes (race, religion) must not be used as features to avoid discriminatory models.

---
## 🌟 Exercise 2 – Feature Selection and Model Choice for Loan Default Prediction

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('ggplot')
pd.set_option('display.max_columns', None)

DATA_URL = 'https://github.com/devtlv/Datasets-GEN-AI-Bootcamp/raw/refs/heads/main/Week%204/Day%201/Loan%20Predication.zip'
DATA_DIR = Path('data')
ZIP_PATH = DATA_DIR / 'loan_prediction.zip'
EXTRACT_DIR = DATA_DIR / 'loan_prediction'

DATA_DIR.mkdir(parents=True, exist_ok=True)

if not ZIP_PATH.exists():
    print('Downloading dataset...')
    urlretrieve(DATA_URL, ZIP_PATH)
    print('Download complete.')

if not EXTRACT_DIR.exists():
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    with ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(EXTRACT_DIR)

csv_files = sorted(EXTRACT_DIR.rglob('*.csv'))
print('CSV files found:', [p.name for p in csv_files])
df = pd.read_csv(csv_files[0])
print(f'\nDataset shape: {df.shape}')
display(df.head())

In [ ]:
print('Column names:', df.columns.tolist())
print('\nData types:')
print(df.dtypes)
print('\nNull values:')
print(df.isna().sum())
print('\nTarget distribution:')
target_col = [c for c in df.columns if 'status' in c.lower() or 'default' in c.lower()]
print(target_col)
if target_col:
    print(df[target_col[0]].value_counts())

### Feature Analysis and Justification

Based on the dataset columns, the following features are expected to be most predictive:

| Feature | Expected Relevance | Justification |
|---|---|---|
| `Credit_History` | **Very High** | Strongest single predictor of default; past behavior predicts future behavior |
| `ApplicantIncome` + `CoapplicantIncome` | **High** | Combined income determines repayment capacity |
| `LoanAmount` | **High** | Higher loan amounts increase default risk for lower-income applicants |
| `Loan_Amount_Term` | **Medium** | Longer terms reduce monthly burden but increase total exposure |
| `Property_Area` | **Medium** | Urban/rural context affects income stability and collateral value |
| `Education` | **Medium** | Correlated with income stability and future earning potential |
| `Self_Employed` | **Medium** | Self-employed income is more volatile than salaried income |
| `Dependents` | **Low-Medium** | More dependents increase financial burden |
| `Gender` / `Married` | **Low** | Weak predictors; use carefully to avoid discriminatory bias |
| `Loan_ID` | **Irrelevant** | Identifier — must be dropped |

**Engineered features worth creating:**
- `Total_Income` = `ApplicantIncome` + `CoapplicantIncome`
- `Debt_Income_Ratio` = `LoanAmount` / `Total_Income`  — key risk metric in lending
- `Monthly_Payment` = `LoanAmount` / `Loan_Amount_Term`

In [ ]:
# Visualize the distribution of key features vs loan status
df_clean = df.copy()

# Standardize column names
df_clean.columns = df_clean.columns.str.strip()

# Feature engineering
if 'ApplicantIncome' in df_clean.columns and 'CoapplicantIncome' in df_clean.columns:
    df_clean['Total_Income'] = df_clean['ApplicantIncome'] + df_clean['CoapplicantIncome']

if 'LoanAmount' in df_clean.columns and 'Total_Income' in df_clean.columns:
    df_clean['Debt_Income_Ratio'] = df_clean['LoanAmount'] / (df_clean['Total_Income'] + 1)

# Plot Credit_History vs Loan_Status if both exist
target = 'Loan_Status' if 'Loan_Status' in df_clean.columns else df_clean.columns[-1]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

if 'Credit_History' in df_clean.columns:
    df_clean.groupby(['Credit_History', target]).size().unstack().plot(
        kind='bar', ax=axes[0], colormap='Set2'
    )
    axes[0].set_title('Credit History vs Loan Status')
    axes[0].set_xlabel('Credit History (0=Bad, 1=Good)')
    axes[0].tick_params(axis='x', rotation=0)

if 'LoanAmount' in df_clean.columns:
    for label, grp in df_clean.groupby(target)['LoanAmount']:
        axes[1].hist(grp.dropna(), bins=30, alpha=0.6, label=str(label))
    axes[1].set_title('Loan Amount Distribution by Status')
    axes[1].set_xlabel('Loan Amount')
    axes[1].legend()

plt.tight_layout()
plt.show()

---
## 🌟 Exercise 3 – Training, Evaluating, and Optimizing the Model

### Model Choices for Loan Prediction

Three models are appropriate for this binary classification problem:

#### 1. Logistic Regression (Baseline)
- **Why:** Fast, interpretable, handles mixed feature types well. Outputs calibrated probabilities useful for setting decision thresholds.
- **Limitation:** Assumes linear decision boundary; may underfit complex relationships.

#### 2. Random Forest Classifier (Primary)
- **Why:** Handles non-linear interactions, robust to outliers, works well with missing data strategies, naturally ranks feature importance.
- **Limitation:** Less interpretable than logistic regression; can overfit on small datasets.

#### 3. Gradient Boosting (e.g., XGBoost) (Advanced)
- **Why:** State-of-the-art on tabular data, handles class imbalance with `scale_pos_weight`, excellent predictive performance.
- **Limitation:** More hyperparameters to tune; slower to train.

---

### Evaluation Strategy

**Step 1 – Train/Validation/Test split**  
Split: 70% train / 15% validation / 15% test (stratified on target to preserve class balance).

**Step 2 – Cross-validation**  
Use Stratified K-Fold (k=5) on the training set to get stable performance estimates and detect overfitting.

**Step 3 – Metrics (binary classification with class imbalance)**

| Metric | Why it matters for loan default |
|---|---|
| **Precision** | Of all predicted defaults, how many actually defaulted? Reduces false alarms (rejected good applicants) |
| **Recall (Sensitivity)** | Of all actual defaults, how many did we catch? Critical — missing a default is costly |
| **F1-Score** | Harmonic mean of precision and recall; good summary when classes are imbalanced |
| **ROC-AUC** | Measures discriminative power across all thresholds; threshold-independent |
| **PR-AUC** | More informative than ROC-AUC when the positive class (default) is rare |
| **Confusion Matrix** | Full breakdown of TP, FP, FN, TN for business decision-making |

**Step 4 – Threshold tuning**  
The default 0.5 threshold is rarely optimal. Adjust based on business cost: if missing a default is more costly than a false alarm, lower the threshold to increase recall.

**Step 5 – Hyperparameter optimization**  
Use `GridSearchCV` or `RandomizedSearchCV` on validation set. Key parameters:
- Random Forest: `n_estimators`, `max_depth`, `min_samples_split`, `class_weight`
- XGBoost: `learning_rate`, `max_depth`, `n_estimators`, `scale_pos_weight`

**Step 6 – Final evaluation on held-out test set**  
Only evaluate on the test set once, after all tuning decisions are finalized.

---
## 🌟 Exercise 4 – Designing Machine Learning Solutions for Specific Problems

### Scenario 1 – Predicting Stock Prices

**Best approach: Supervised Learning — Regression (e.g., LSTM / Time-Series Regression)**

**Justification:**
- The task is to predict a **continuous numerical value** (future price) from historical data → this is a regression problem within supervised learning.
- Stock prices are sequential time-series data with temporal dependencies; models like **LSTM (Long Short-Term Memory)** networks or **ARIMA/Prophet** are well-suited because they capture patterns over time.
- Classical supervised regressors (Random Forest Regressor, Ridge Regression) can also work if features are carefully engineered (lagged values, moving averages, volume trends).
- **Why not unsupervised?** We have labeled historical prices, so supervision is possible and more powerful.
- **Why not reinforcement learning?** RL is better suited for *trading strategy* (buy/sell/hold decisions) rather than raw price prediction.

---

### Scenario 2 – Organizing a Library of Books

**Best approach: Unsupervised Learning — Clustering (e.g., K-Means, Hierarchical Clustering, LDA)**

**Justification:**
- The task is to **discover natural groupings** (genres/categories) among books without pre-labeled categories → this is exactly the use case for **unsupervised clustering**.
- Text features (book descriptions, titles, keywords) can be vectorized with TF-IDF or sentence embeddings, then clustered.
- **Topic modeling (LDA)** is another unsupervised approach that discovers latent themes across documents.
- **K-Means** works well when the number of genres is approximately known; **Hierarchical Clustering** is better for exploring unknown structure.
- **Why not supervised?** Supervised classification would require pre-labeled genre data, which may not exist for new collections.

---

### Scenario 3 – Robot Navigation in a Maze

**Best approach: Reinforcement Learning (e.g., Q-Learning, Deep Q-Network)**

**Justification:**
- The robot must **learn by trial and error** through interaction with the environment (the maze) — no labeled training data of correct paths is available.
- **Q-Learning** assigns reward values to state-action pairs; the robot learns the optimal policy (shortest path) by maximizing cumulative reward.
- The reward structure is natural: +large reward for reaching the exit, -small penalty per step (encourages shortest path), -large penalty for hitting walls.
- **Why not supervised?** We don't have labeled examples of correct navigation decisions for every maze state.
- **Why not classical algorithms?** While BFS/Dijkstra solve mazes deterministically, RL generalizes to *unknown, dynamic environments* where the map changes or obstacles appear.

---

### Summary Table

| Scenario | ML Paradigm | Recommended Algorithm | Key Reason |
|---|---|---|---|
| Stock price prediction | Supervised – Regression | LSTM / Time-series regression | Labeled sequential data, continuous output |
| Library organization | Unsupervised – Clustering | K-Means / LDA | No labels; discover natural groups |
| Robot maze navigation | Reinforcement Learning | Q-Learning / DQN | Trial-and-error in interactive environment |

---
## 🌟 Exercise 5 – Designing an Evaluation Strategy for Different ML Models

### Model 1 – Supervised Learning: Classification (Random Forest for Loan Default)

**Evaluation strategy:**

1. **Train/test split** with stratification to preserve class proportions.
2. **Stratified K-Fold Cross-Validation (k=5)** to get unbiased performance estimates across different data subsets.
3. **Metrics:**
   - *Accuracy:* Baseline but misleading when classes are imbalanced.
   - *Precision & Recall:* Trade-off depends on business cost of false positives vs false negatives.
   - *F1-Score:* Single metric balancing precision and recall.
   - *ROC-AUC:* Measures the model's ability to discriminate between classes at all thresholds.
   - *Confusion Matrix:* Provides full insight into TP, FP, TN, FN counts.
4. **ROC Curve** — plot True Positive Rate vs False Positive Rate for different decision thresholds.
5. **Challenges:** Class imbalance (fewer defaults than approvals) can inflate accuracy; use SMOTE or `class_weight='balanced'` to address it.

---

### Model 2 – Unsupervised Learning: Clustering (K-Means for Library Organization)

**Evaluation strategy:**

1. **Elbow Method:** Plot inertia (within-cluster sum of squares) vs number of clusters k. The "elbow" point suggests the optimal k where adding more clusters yields diminishing returns.
2. **Silhouette Score:** Measures how similar each point is to its own cluster vs neighboring clusters. Score ranges from -1 (wrong cluster) to +1 (dense, well-separated cluster). Target: > 0.5.
3. **Davies-Bouldin Index:** Lower values indicate better separation between clusters.
4. **Calinski-Harabasz Index (Variance Ratio Criterion):** Higher values indicate more compact and well-separated clusters.
5. **Qualitative validation:** Inspect cluster contents — do books in the same cluster share coherent themes? Domain expert review is essential.
6. **Challenges:** No ground truth labels make purely quantitative evaluation insufficient. The "best" number of clusters depends on domain needs, not just mathematical metrics. High-dimensional text data requires dimensionality reduction (PCA, UMAP) for meaningful clustering.

---

### Model 3 – Reinforcement Learning: Q-Learning (Robot Maze Navigation)

**Evaluation strategy:**

1. **Cumulative Reward:** Track total reward per episode over training. A well-learning agent should show increasing cumulative reward over time as it discovers better paths.
2. **Convergence:** Monitor the Q-table (or Q-network) updates. When values stabilize (delta Q ≈ 0), the policy has converged to an optimal or near-optimal solution.
3. **Average Steps to Goal:** After training, run evaluation episodes (no exploration) and measure the average number of steps to reach the exit. Compare with the optimal path length (from BFS).
4. **Exploration vs Exploitation balance (ε-greedy):** Track the epsilon decay curve. Too much exploration wastes time; too little prevents finding better paths. Evaluate final policy with ε=0 (pure exploitation).
5. **Success Rate:** Percentage of evaluation episodes where the robot reaches the exit (vs getting stuck in loops or time-out).
6. **Challenges:**
   - **Reward shaping:** Poorly designed rewards can lead to unexpected behaviors (e.g., robot wandering to avoid the -step penalty).
   - **Generalization:** A Q-table trained on one maze doesn't generalize to new mazes — Deep Q-Networks (DQN) partially address this by learning transferable features.
   - **Stability:** RL training can be unstable; results vary significantly between runs without fixed seeds.

---

### Comparative Summary

| Aspect | Supervised (Classification) | Unsupervised (Clustering) | Reinforcement Learning |
|---|---|---|---|
| Ground truth available? | Yes (labels) | No | No (reward signal only) |
| Primary metrics | Accuracy, F1, ROC-AUC | Silhouette, Elbow, Davies-Bouldin | Cumulative reward, steps to goal |
| Validation method | K-Fold CV, train/test split | Mathematical indices + domain review | Evaluation episodes with ε=0 |
| Main challenge | Class imbalance, overfitting | No labels for quantitative truth | Reward design, stability, generalization |

---
## 📝 Summary

This notebook covered:
1. **Problem definition** and data collection plan for loan default prediction, including data types (credit history, income, loan details) and sources (internal records, credit bureaus).
2. **Feature selection** from the Loan Prediction dataset — `Credit_History`, `Total_Income`, and `Debt_Income_Ratio` identified as most predictive.
3. **Model selection and evaluation** — Logistic Regression (baseline), Random Forest (primary), and XGBoost (advanced); evaluation via F1, ROC-AUC, and stratified cross-validation.
4. **ML paradigm selection** — Supervised regression for stock prices, Unsupervised clustering for library organization, Reinforcement learning for robot maze navigation.
5. **Evaluation strategies** for supervised (cross-validation, ROC curves), unsupervised (silhouette score, elbow method), and reinforcement learning (cumulative reward, convergence).